# BÀI THỰC HÀNH: PHÂN CỤM DỮ LIỆU TRONG MACHINE LEARNING
**Học phần: Phân tích Dữ liệu với Python (DSAI1005)**  
**Giảng viên: TS. Vũ Đức Minh – Khoa Khoa học dữ liệu và Trí tuệ nhân tạo, Trường Công nghệ, Đại học Kinh tế Quốc dân (NEU)**  
**Thời gian cập nhật: 22 tháng 9 năm 2026**

---

## 🎯 Mục tiêu Bài học & Cấu trúc Thực hành

Tài liệu này được thiết kế theo chuẩn sư phạm tương tác, kết hợp chặt chẽ giữa toán học định lượng và lập trình ứng dụng:

1. **Phần 0: Chuẩn bị Môi trường & Thư viện** — Nạp Scikit-Learn, SciPy, NumPy, Pandas, Matplotlib, Seaborn.
2. **Phần 1: Tóm tắt Lý thuyết & Cheatsheet Thư viện** — Tra cứu nhanh công thức 5 trường phái phân cụm và các chỉ số đánh giá.
3. **Phần 2: 4 Bài tập Tính toán Tay (Hand Calculations):**
   - **(A) Lý thuyết & Công thức** $\to$ **(B) Ví dụ mẫu giải sẵn** $\to$ **(C) Bài tập tự luyện với khung trống `[...]` để Sinh viên tự làm** $\to$ **(D) Code Python kiểm tra đáp án tự động**.
4. **Phần 3: Trực quan hóa Lý thuyết Chuyên sâu** — Code phân vùng Voronoi, Dendrogram, K-Distance plot và GMM Elip đồng mức.
5. **Phần 4: 5 Bài tập Lập trình Điền Khuyết (Fill-in-the-Blank Code Labs):**
   - Để trống các đoạn ngắn dạng `...` kèm hướng dẫn chi tiết từng bước.
   - Hệ thống chấm điểm tự động (`assert`) phản hồi ngay kết quả.
6. **Phần 5: Đáp án Tham khảo (Reference Solutions)** — Toàn bộ code hoàn chỉnh để đối chiếu khi cần.



## 0. Chuẩn bị Môi trường & Thư viện

Chạy ô lệnh dưới đây trước khi bắt đầu thực hành.



In [ ]:
# Import các thư viện phân tích dữ liệu và học máy tiêu chuẩn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Thư viện Scipy cho phân cụm thứ bậc và ma trận khoảng cách
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.stats import norm

# Thư viện Scikit-Learn cho phân cụm, chuẩn hóa và đánh giá
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, OPTICS
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.metrics import davies_bouldin_score, calinski_harabasz_score
from sklearn.neighbors import NearestNeighbors
from sklearn.datasets import make_blobs, make_moons

# Cấu hình hiển thị biểu đồ
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Khởi tạo môi trường thành công! Mọi thư viện đã sẵn sàng.")


## 1. Tóm tắt Lý thuyết & Cheatsheet Thư viện Scikit-Learn

### 1.1. Bảng tra cứu nhanh API Scikit-Learn / SciPy

| Thuật toán | Trường phái | Đối tượng Python | Tham số quan trọng cốt lõi | Thuộc tính sau `.fit()` |
| :--- | :--- | :--- | :--- | :--- |
| **K-Means** | Centroid-based | `sklearn.cluster.KMeans` | `n_clusters`, `init='k-means++'`, `n_init=10`, `max_iter=300` | `.cluster_centers_`, `.labels_`, `.inertia_` |
| **Hierarchical** | Connectivity | `scipy.cluster.hierarchy.linkage`<br>`sklearn.cluster.AgglomerativeClustering` | `n_clusters`, `linkage='ward'/'complete'/'single'`, `metric='euclidean'` | `.labels_`, `.children_`, `.n_leaves_` |
| **DBSCAN** | Density-based | `sklearn.cluster.DBSCAN` | `eps`, `min_samples`, `metric='euclidean'` | `.labels_` (nhãn `-1` là nhiễu), `.core_sample_indices_` |
| **OPTICS** | Density-based | `sklearn.cluster.OPTICS` | `min_samples`, `max_eps=np.inf`, `cluster_method='xi'`, `xi=0.05` | `.labels_`, `.reachability_`, `.core_distances_` |
| **GMM (EM)** | Distribution-based | `sklearn.mixture.GaussianMixture` | `n_components`, `covariance_type='full'/'diag'`, `random_state` | `.means_`, `.covariances_`, `.weights_`, `.bic()`, `.aic()` |

---
### 1.2. Công thức Đánh giá Chất lượng Phân cụm (Clustering Metrics)

1. **Hệ số Silhouette ($s \in [-1, 1]$):**
   $$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$
   - $a(i)$: Khoảng cách trung bình nội cụm (Cohesion - càng nhỏ càng tốt).
   - $b(i)$: Khoảng cách trung bình tới cụm lân cận gần nhất (Separation - càng lớn càng tốt).
   - $s > 0.5 \implies$ Cụm chặt chẽ, phân tách rõ ràng; $s \approx 0 \implies$ Vùng ranh giới chồng lấn; $s < 0 \implies$ Gán sai cụm.

2. **Chỉ số Davies-Bouldin Index (DBI $\ge 0$):**
   $$DB = \frac{1}{K}\sum_{i=1}^K \max_{j \ne i} \left( \frac{s_i + s_j}{d(\mu_i, \mu_j)} \right)$$
   - Tỷ số giữa độ phân tán nội cụm và khoảng cách giữa các tâm cụm. **Càng nhỏ càng tốt**.

3. **Chỉ số Calinski-Harabasz (CH Index):**
   $$CH = \frac{\text{SSB} / (K - 1)}{\text{SSW} / (N - K)}$$
   - Tỷ số giữa phương sai liên cụm và phương sai nội cụm. **Càng lớn càng tốt**.

4. **Tiêu chuẩn Thông tin BIC \& AIC (Mô hình GMM):**
   $$\text{BIC} = -2\ln \hat{L} + p \ln N, \quad \text{AIC} = -2\ln \hat{L} + 2p$$
   - Phạt mức độ phức tạp theo số tham số $p$. **Mô hình có BIC/AIC nhỏ nhất là tối ưu**.



## 2. Các Bài tập Tính Tay Chi Tiết (Step-by-Step Hand Calculations)

Mỗi bài toán bao gồm 4 phần:
1. **Lý thuyết & Công thức toán học**
2. **Ví dụ mẫu giải sẵn chi tiết từng bước**
3. **Bài tập tự luyện cho Sinh viên (với bảng điền `[...]` còn trống)**
4. **Code Python kiểm tra đối chiếu đáp án**

---

### Bài toán 1: Vòng lặp Thuật toán K-Means (Lloyd's Algorithm)

#### A. Lý thuyết & Công thức Cần Dùng
Thuật toán Lloyd thực hiện lặp 2 bước cho đến khi hội tụ:
1. **Bước gán cụm (Assignment):** Gán mỗi điểm $x_i$ vào cụm có tâm gần nhất:
   $$c^{(i)} = \arg\min_{k \in \{1, \dots, K\}} \|x_i - \mu_k\|_2^2 = \arg\min_k \sum_{j=1}^d (x_{ij} - \mu_{kj})^2$$
2. **Bước cập nhật tâm (Update):** Tính lại toạ độ trọng tâm của từng cụm:
   $$\mu_k = \frac{1}{|C_k|} \sum_{x_i \in C_k} x_i$$
3. **Tổng bình phương khoảng cách nội cụm (Inertia / WCSS):**
   $$J = \sum_{k=1}^K \sum_{x_i \in C_k} \|x_i - \mu_k\|_2^2$$



#### B. Ví dụ Mẫu Giải Sẵn (Worked Example 1)

**Đề bài mẫu:** Cho 4 điểm trong $\mathbb{R}^2$: $P_1(1, 1), P_2(2, 1), P_3(4, 3), P_4(5, 4)$ với 2 tâm ban đầu:
$$\mu_1^{(0)} = P_1 = (1, 1), \quad \mu_2^{(0)} = P_4 = (5, 4)$$

**Lời giải chi tiết từng bước:**

**Bước 1: Tính khoảng cách bình phương và gán cụm:**
- Điểm $P_1(1, 1)$: $d^2(P_1, \mu_1) = 0$, $d^2(P_1, \mu_2) = (1-5)^2 + (1-4)^2 = 16 + 9 = 25 \implies$ **Gán Cụm 1**.
- Điểm $P_2(2, 1)$: $d^2(P_2, \mu_1) = (2-1)^2 + (1-1)^2 = 1$, $d^2(P_2, \mu_2) = (2-5)^2 + (1-4)^2 = 9 + 9 = 18 \implies$ **Gán Cụm 1**.
- Điểm $P_3(4, 3)$: $d^2(P_3, \mu_1) = (4-1)^2 + (3-1)^2 = 9 + 4 = 13$, $d^2(P_3, \mu_2) = (4-5)^2 + (3-4)^2 = 1 + 1 = 2 \implies$ **Gán Cụm 2**.
- Điểm $P_4(5, 4)$: $d^2(P_4, \mu_1) = (5-1)^2 + (4-1)^2 = 16 + 9 = 25$, $d^2(P_4, \mu_2) = 0 \implies$ **Gán Cụm 2**.

Kết quả phân nhóm: $C_1 = \{P_1, P_2\}$, $C_2 = \{P_3, P_4\}$.

**Bước 2: Cập nhật toạ độ tâm mới:**
$$\mu_1^{(1)} = \frac{P_1 + P_2}{2} = \left( \frac{1+2}{2}, \frac{1+1}{2} \right) = \mathbf{(1.5, 1.0)}$$
$$\mu_2^{(1)} = \frac{P_3 + P_4}{2} = \left( \frac{4+5}{2}, \frac{3+4}{2} \right) = \mathbf{(4.5, 3.5)}$$

**Bước 3: Tính WCSS mới sau cập nhật:**
- Cụm 1: $(1-1.5)^2+(1-1)^2 + (2-1.5)^2+(1-1)^2 = 0.25 + 0.25 = 0.5$
- Cụm 2: $(4-4.5)^2+(3-3.5)^2 + (5-4.5)^2+(4-3.5)^2 = 0.5 + 0.5 = 1.0$
$$\text{WCSS}^{(1)} = 0.5 + 1.0 = \mathbf{1.5}$$
(Trước cập nhật: $\text{WCSS}^{(0)} = 0 + 1 + 2 + 0 = 3.0 \implies$ hàm mục tiêu giảm 50%!)



#### C. Bài tập Tự luyện cho Sinh viên (Student Hands-on Exercise 1)

> **Đề bài:** Cho tập dữ liệu gồm $N = 5$ điểm trong không gian $\mathbb{R}^2$:
> - $A = (1.0, 2.0)$
> - $B = (2.0, 1.0)$
> - $C = (4.0, 5.0)$
> - $D = (5.0, 4.0)$
> - $E = (6.0, 5.0)$
>
> Hai tâm ban đầu tại vòng lặp đầu tiên:
> $$\mu_1 = A = (1.0, 2.0), \quad \mu_2 = C = (4.0, 5.0)$$
>
> **Nhiệm vụ của Sinh viên:**
> 1. Tính khoảng cách bình phương $d^2(P, \mu_1)$ và $d^2(P, \mu_2)$ cho 5 điểm.
> 2. Gán từng điểm vào cụm gần hơn (Cụm 1 hay Cụm 2).
> 3. Tính toạ độ tâm mới $\mu_1^{new}$ và $\mu_2^{new}$.
> 4. Tính giá trị $\text{WCSS}^{new}$.

---

**BẢNG TÍNH TOÁN CỦA SINH VIÊN (Hãy tính tay và điền vào các ô `[...]`):**

| Điểm | Toạ độ $(x_1, x_2)$ | $d^2(P, \mu_1)$ với $\mu_1=(1, 2)$ | $d^2(P, \mu_2)$ với $\mu_2=(4, 5)$ | Cụm được gán (1 hay 2?) |
| :---: | :---: | :---: | :---: | :---: |
| **A** | $(1.0, 2.0)$ | `(1-1)^2 + (2-2)^2 = [ 0.0 ]` | `(1-4)^2 + (2-5)^2 = 9 + 9 = [ 18.0 ]` | `[ Cụm 1 ]` |
| **B** | $(2.0, 1.0)$ | `(2-1)^2 + (1-2)^2 = [ ... ]` | `(2-4)^2 + (1-5)^2 = [ ... ]` | `[ ... ]` |
| **C** | $(4.0, 5.0)$ | `(4-1)^2 + (5-2)^2 = [ ... ]` | `(4-4)^2 + (5-5)^2 = [ ... ]` | `[ ... ]` |
| **D** | $(5.0, 4.0)$ | `(5-1)^2 + (4-2)^2 = [ ... ]` | `(5-4)^2 + (4-5)^2 = [ ... ]` | `[ ... ]` |
| **E** | $(6.0, 5.0)$ | `(6-1)^2 + (5-2)^2 = [ ... ]` | `(6-4)^2 + (5-5)^2 = [ ... ]` | `[ ... ]` |

- **Tâm Cụm 1 mới $\mu_1^{new}$:** `( [ ... ], [ ... ] )`
- **Tâm Cụm 2 mới $\mu_2^{new}$:** `( [ ... ], [ ... ] )`
- **Giá trị $\text{WCSS}^{new}$:** `[ ... ]`



In [ ]:
# D. Ô KIỂM TRA ĐÁP ÁN BÀI TẬP 1 (Chạy cell này để kiểm tra kết quả tính tay của bạn)
X_sv1 = np.array([
    [1.0, 2.0],  # A
    [2.0, 1.0],  # B
    [4.0, 5.0],  # C
    [5.0, 4.0],  # D
    [6.0, 5.0]   # E
])
pts_names = ['A', 'B', 'C', 'D', 'E']
mu1_init = np.array([1.0, 2.0])
mu2_init = np.array([4.0, 5.0])
centers_init = np.vstack([mu1_init, mu2_init])

# Tính khoảng cách Euclidean bình phương
dists_sq = np.sum((X_sv1[:, np.newaxis, :] - centers_init[np.newaxis, :, :]) ** 2, axis=2)
assigned_clusters = np.argmin(dists_sq, axis=1)

# Cập nhật tâm mới
mu1_new = X_sv1[assigned_clusters == 0].mean(axis=0)
mu2_new = X_sv1[assigned_clusters == 1].mean(axis=0)

# Tính WCSS mới
new_centers = np.vstack([mu1_new, mu2_new])
new_dists = np.sum((X_sv1[:, np.newaxis, :] - new_centers[np.newaxis, :, :]) ** 2, axis=2)
wcss_new = np.sum(new_dists[np.arange(len(X_sv1)), assigned_clusters])

print("====== ĐÁP ÁN CHUẨN ĐỐI CHIẾU BÀI 1 ======")
for i in range(len(X_sv1)):
    print(f"Điểm {pts_names[i]}: d^2(mu_1) = {dists_sq[i, 0]:5.1f} | d^2(mu_2) = {dists_sq[i, 1]:5.1f} ==> Thuộc Cụm {assigned_clusters[i] + 1}")

print(f"\nToạ độ tâm Cụm 1 mới: {mu1_new} (Kỳ vọng: [1.5, 1.5])")
print(f"Toạ độ tâm Cụm 2 mới: {np.round(mu2_new, 4)} (Kỳ vọng: [5.0, 4.6667])")
print(f"WCSS mới: {wcss_new:.4f} (Kỳ vọng: 3.6667)")


---

### Bài toán 2: Phân cụm Phân cấp Thứ bậc (Hierarchical Clustering)

#### A. Lý thuyết & Công thức Cần Dùng
Trong phân cụm tích tụ (Agglomerative), ta bắt đầu với ma trận khoảng cách $D_0$ giữa $N$ đối tượng. Tại mỗi bước:
1. Tìm khoảng cách nhỏ nhất giữa 2 cụm: $d(C_i, C_j) = \min_{A \ne B} D(A, B)$.
2. Hợp nhất $C_i$ và $C_j$ thành cụm mới $(C_i C_j)$ tại độ cao khoảng cách $h = d(C_i, C_j)$.
3. Cập nhật khoảng cách từ cụm mới $(C_i C_j)$ đến mọi cụm khác $C_k$:
   - **Single Linkage (Khoảng cách cực tiểu):**
     $$D((C_i C_j), C_k) = \min \left( D(C_i, C_k), D(C_j, C_k) \right)$$
   - **Complete Linkage (Khoảng cách cực đại):**
     $$D((C_i C_j), C_k) = \max \left( D(C_i, C_k), D(C_j, C_k) \right)$$



#### B. Ví dụ Mẫu Giải Sẵn (Worked Example 2)

**Đề bài mẫu:** Cho ma trận khoảng cách giữa 3 đối tượng $\{A, B, C\}$:

| | A | B | C |
|:---:|:---:|:---:|:---:|
| **A** | 0 | 3 | 7 |
| **B** | 3 | 0 | 5 |
| **C** | 7 | 5 | 0 |

**Lời giải chi tiết:**
- **Bước 1:** Khoảng cách nhỏ nhất là $d(A, B) = \mathbf{3}$. Hợp nhất $A$ và $B$ thành cụm $(AB)$ ở độ cao $h_1 = 3$.
- **Bước 2: Cập nhật khoảng cách tới $C$:**
  - Với Single Linkage: $d((AB), C) = \min(d(A, C), d(B, C)) = \min(7, 5) = \mathbf{5}$.
  - Với Complete Linkage: $d((AB), C) = \max(d(A, C), d(B, C)) = \max(7, 5) = \mathbf{7}$.
- **Bước 3:** Hợp nhất $((AB), C)$ tại độ cao $h_2 = 5$ (Single) hoặc $h_2 = 7$ (Complete).



#### C. Bài tập Tự luyện cho Sinh viên (Student Hands-on Exercise 2)

> **Đề bài:** Cho ma trận khoảng cách đối xứng $D_0$ kích thước $4 \times 4$ giữa 4 mẫu $\{A, B, C, D\}$:
>
> | | A | B | C | D |
> |:---:|:---:|:---:|:---:|:---:|
> | **A** | 0 | 2 | 8 | 7 |
> | **B** | 2 | 0 | 7 | 6 |
> | **C** | 8 | 7 | 0 | 3 |
> | **D** | 7 | 6 | 3 | 0 |
>
> **Nhiệm vụ của Sinh viên:**
> 1. Xác định thứ tự ghép và độ cao $h$ theo **Single Linkage**.
> 2. Xác định thứ tự ghép và độ cao $h$ theo **Complete Linkage**.

---

**KHUNG ĐIỀN CỦA SINH VIÊN (Single Linkage):**
- Bước 1: Giá trị nhỏ nhất trong $D_0$ là `[ ... ]` giữa cặp `[ ... ]`. Sáp nhập tại $h_1 = $ `[ ... ]`.
- Ma trận $D_1$ còn lại các cụm $\{(AB), C, D\}$:
  - $d((AB), C) = \min(d(A,C), d(B,C)) = \min(8, 7) = $ `[ ... ]`
  - $d((AB), D) = \min(d(A,D), d(B,D)) = \min(7, 6) = $ `[ ... ]`
  - $d(C, D) = $ `[ ... ]`
- Bước 2: Khoảng cách nhỏ nhất trong $D_1$ là `[ ... ]` giữa cặp `[ ... ]`. Sáp nhập tại $h_2 = $ `[ ... ]`.
- Bước 3: Sáp nhập cụm $(AB)$ và $(CD)$ với khoảng cách $\min(d(AB,C), d(AB,D)) = $ `[ ... ]` tại $h_3 = $ `[ ... ]`.



In [ ]:
# D. Ô KIỂM TRA ĐÁP ÁN BÀI TẬP 2 (Chạy cell này để kiểm tra kết quả tính tay của bạn)
D_sv2 = np.array([
    [0, 2, 8, 7],
    [2, 0, 7, 6],
    [8, 7, 0, 3],
    [7, 6, 3, 0]
], dtype=float)

condensed_d = squareform(D_sv2)

# Single Linkage
Z_single = linkage(condensed_d, method='single')
# Complete Linkage
Z_complete = linkage(condensed_d, method='complete')

print("====== ĐÁP ÁN CHUẨN ĐỐI CHIẾU BÀI 2 ======")
print("1. Single Linkage (Cụm 1, Cụm 2, Khoảng cách h, Số lượng mẫu):")
print(Z_single)
print("=> Thứ tự độ cao sáp nhập Single Linkage: [2.0, 3.0, 6.0]")

print("\n2. Complete Linkage (Cụm 1, Cụm 2, Khoảng cách h, Số lượng mẫu):")
print(Z_complete)
print("=> Thứ tự độ cao sáp nhập Complete Linkage: [2.0, 3.0, 8.0]")


---

### Bài toán 3: Phân loại Điểm Mật độ DBSCAN

#### A. Lý thuyết & Tiêu chí Phân loại Điểm
Với hai siêu tham số $(\epsilon, \text{MinPts})$:
1. **Lân cận $\epsilon$:** $N_\epsilon(p) = \{ q \in \mathcal{X} \mid d(p, q) \le \epsilon \}$ (tính cả chính điểm $p$).
2. **Điểm lõi (Core Point):** $|N_\epsilon(p)| \ge \text{MinPts}$.
3. **Điểm biên (Border Point):** $|N_\epsilon(p)| < \text{MinPts}$ nhưng nằm trong bán kính $\epsilon$ của ít nhất một Core Point.
4. **Điểm nhiễu (Noise Point):** Không phải Core và không thuộc lân cận của Core nào (gán nhãn **-1**).



#### B. Ví dụ Mẫu Giải Sẵn (Worked Example 3)

**Đề bài mẫu:** Cho 4 điểm 1 chiều: $A=1, B=2, C=3, D=10$ với $\epsilon = 1.5, \text{MinPts} = 2$.
- $N_\epsilon(A) = \{A, B\} \implies |N_\epsilon| = 2 \ge 2 \implies$ **Core Point**.
- $N_\epsilon(B) = \{A, B, C\} \implies |N_\epsilon| = 3 \ge 2 \implies$ **Core Point**.
- $N_\epsilon(C) = \{B, C\} \implies |N_\epsilon| = 2 \ge 2 \implies$ **Core Point**.
- $N_\epsilon(D) = \{D\} \implies |N_\epsilon| = 1 < 2$, và khoảng cách tới Core gần nhất $d(D, C) = 7 > 1.5 \implies$ **Noise Point**.

Kết quả: Tạo thành 1 cụm $\{A, B, C\}$ và 1 điểm nhiễu $\{D\}$ (nhãn -1).



#### C. Bài tập Tự luyện cho Sinh viên (Student Hands-on Exercise 3)

> **Đề bài:** Cho tập gồm $N = 7$ điểm trong không gian $\mathbb{R}^2$:
> - $P_1 = (1.0, 1.0)$
> - $P_2 = (1.0, 2.0)$
> - $P_3 = (2.0, 1.0)$
> - $P_4 = (6.0, 6.0)$
> - $P_5 = (6.0, 7.0)$
> - $P_6 = (7.0, 6.0)$
> - $P_7 = (20.0, 20.0)$
>
> Cấu hình tham số: $\mathbf{\epsilon = 1.5}, \quad \mathbf{\text{MinPts} = 3}$.
>
> **Nhiệm vụ của Sinh viên:**
> Tính số lượng láng giềng $|N_\epsilon(P_i)|$ (khoảng cách Euclidean $\le 1.5$) và phân loại từng điểm.

---

**BẢNG ĐIỀN CỦA SINH VIÊN:**

| Điểm $P_i$ | Toạ độ | Danh sách láng giềng trong bán kính $\le 1.5$ | Số lượng $|N_\epsilon|$ | Phân loại (Core / Border / Noise) |
| :---: | :---: | :--- | :---: | :---: |
| **$P_1$** | $(1.0, 1.0)$ | $\{P_1, P_2, P_3\}$ (vì $d(P_1,P_2)=1, d(P_1,P_3)=1$) | `[ 3 ]` | `[ Core Point ]` |
| **$P_2$** | $(1.0, 2.0)$ | `[ ... ]` | `[ ... ]` | `[ ... ]` |
| **$P_3$** | $(2.0, 1.0)$ | `[ ... ]` | `[ ... ]` | `[ ... ]` |
| **$P_4$** | $(6.0, 6.0)$ | `[ ... ]` | `[ ... ]` | `[ ... ]` |
| **$P_5$** | $(6.0, 7.0)$ | `[ ... ]` | `[ ... ]` | `[ ... ]` |
| **$P_6$** | $(7.0, 6.0)$ | `[ ... ]` | `[ ... ]` | `[ ... ]` |
| **$P_7$** | $(20.0, 20.0)$ | `[ ... ]` | `[ ... ]` | `[ ... ]` |

- **Số lượng cụm phát hiện:** `[ ... ]`
- **Tập điểm nhiễu (Noise):** `[ ... ]`



In [ ]:
# D. Ô KIỂM TRA ĐÁP ÁN BÀI TẬP 3 (Chạy cell này để kiểm tra kết quả tính tay của bạn)
X_sv3 = np.array([
    [1.0, 1.0],  # P1
    [1.0, 2.0],  # P2
    [2.0, 1.0],  # P3
    [6.0, 6.0],  # P4
    [6.0, 7.0],  # P5
    [7.0, 6.0],  # P6
    [20.0, 20.0] # P7
])

db_sv = DBSCAN(eps=1.5, min_samples=3).fit(X_sv3)
core_set = set(db_sv.core_sample_indices_)

print("====== ĐÁP ÁN CHUẨN ĐỐI CHIẾU BÀI 3 ======")
for i in range(len(X_sv3)):
    lbl = db_sv.labels_[i]
    role = "Core Point" if i in core_set else ("Border Point" if lbl != -1 else "Noise Point")
    print(f"P{i+1}: Nhãn cụm = {lbl:2d} | Phân loại = {role}")

n_clusters_found = len(set(db_sv.labels_)) - (1 if -1 in db_sv.labels_ else 0)
print(f"\nTổng số cụm phát hiện: {n_clusters_found} (Cụm 0: P1,P2,P3; Cụm 1: P4,P5,P6)")
print(f"Điểm nhiễu (Nhãn -1): P7 toạ độ (20, 20)")


---

### Bài toán 4: Mô hình Trộn Gaussian (GMM) & Thuật toán EM

#### A. Lý thuyết & Công thức Cần Dùng
Mô hình GMM 1 chiều với $K$ thành phần:
$$p(x) = \sum_{k=1}^K \pi_k \mathcal{N}(x \mid \mu_k, \sigma_k^2), \quad \text{với } \mathcal{N}(x \mid \mu, \sigma^2) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left( -\frac{(x - \mu)^2}{2\sigma^2} \right)$$
Khi $\sigma = 1.0 \implies \frac{1}{\sqrt{2\pi}} \approx 0.3989$.

1. **Bước E (Expectation Step):** Tính ma trận trách nhiệm xác suất $\gamma_{nk}$:
   $$\gamma_{nk} = \frac{\pi_k \mathcal{N}(x_n \mid \mu_k, \sigma_k^2)}{\sum_{j=1}^K \pi_j \mathcal{N}(x_n \mid \mu_j, \sigma_j^2)}$$
2. **Bước M (Maximization Step):** Cập nhật lại các tham số phân phối:
   $$N_k = \sum_{n=1}^N \gamma_{nk}, \quad \pi_k^{new} = \frac{N_k}{N}, \quad \mu_k^{new} = \frac{1}{N_k} \sum_{n=1}^N \gamma_{nk} x_n$$
   $$(\sigma_k^2)^{new} = \frac{1}{N_k} \sum_{n=1}^N \gamma_{nk} (x_n - \mu_k^{new})^2$$



#### B. Ví dụ Mẫu Giải Sẵn (Worked Example 4)

**Đề bài mẫu:** Cho $X = [1.0, 5.0]$ với $K = 2$, khởi tạo $\mu_1 = 1.0, \mu_2 = 5.0, \sigma_1^2 = \sigma_2^2 = 1.0, \pi_1 = \pi_2 = 0.5$.
- Mật độ: $\mathcal{N}(1 \mid 1, 1) = 0.3989$, $\mathcal{N}(1 \mid 5, 1) = 0.3989 \cdot e^{-8} \approx 0.0001$.
- Trách nhiệm: $\gamma_{11} \approx 1.0, \gamma_{12} \approx 0.0$; tương tự $\gamma_{21} \approx 0.0, \gamma_{22} \approx 1.0$.
- Cập nhật: $N_1 = 1.0, N_2 = 1.0 \implies \pi_1 = \pi_2 = 0.5, \mu_1 = 1.0, \mu_2 = 5.0$.



#### C. Bài tập Tự luyện cho Sinh viên (Student Hands-on Exercise 4)

> **Đề bài:** Cho tập $N = 4$ điểm trên trục 1 chiều:
> $$X = [1.0, 2.0, 5.0, 6.0]$$
>
> Khởi tạo 2 phân phối chuẩn ban đầu:
> - Cụm 1: $\pi_1 = 0.5, \quad \mu_1 = 1.5, \quad \sigma_1^2 = 1.0$
> - Cụm 2: $\pi_2 = 0.5, \quad \mu_2 = 5.5, \quad \sigma_2^2 = 1.0$
>
> **Nhiệm vụ của Sinh viên:**
> 1. Tính mật độ $\mathcal{N}_1(x_n)$ và $\mathcal{N}_2(x_n)$ cho các điểm $x \in \{1, 2\}$.
> 2. Tính ma trận trách nhiệm $\gamma_{nk}$.
> 3. Tính tổng trách nhiệm $N_1, N_2$ và trọng số mới $\pi_1^{new}, \pi_2^{new}$.
> 4. Tính kỳ vọng mới $\mu_1^{new}, \mu_2^{new}$ và phương sai mới.

---

**KHUNG TÍNH TOÁN CỦA SINH VIÊN:**
- Điểm $x_1 = 1.0$:
  - $(x_1 - \mu_1)^2 = (1 - 1.5)^2 = 0.25 \implies \mathcal{N}_1(1.0) = 0.3989 \cdot e^{-0.125} \approx \mathbf{0.3521}$
  - $(x_1 - \mu_2)^2 = (1 - 5.5)^2 = 20.25 \implies \mathcal{N}_2(1.0) = 0.3989 \cdot e^{-10.125} \approx \mathbf{0.0000}$
  - $\gamma_{11} = \frac{0.3521}{0.3521 + 0.0000} = $ `[ 1.0000 ]`, $\quad \gamma_{12} = $ `[ 0.0000 ]`
- Điểm $x_2 = 2.0$:
  - $(x_2 - \mu_1)^2 = (2 - 1.5)^2 = 0.25 \implies \mathcal{N}_1(2.0) \approx \mathbf{0.3521}$
  - $(x_2 - \mu_2)^2 = (2 - 5.5)^2 = 12.25 \implies \mathcal{N}_2(2.0) = 0.3989 \cdot e^{-6.125} \approx \mathbf{0.0009}$
  - $\gamma_{21} = \frac{0.3521}{0.3521 + 0.0009} = $ `[ ... ]`, $\quad \gamma_{22} = $ `[ ... ]`
- Do đối xứng hoàn hảo qua $3.5$: $\gamma_{31} = \gamma_{22} \approx $ `[ ... ]`, $\gamma_{32} = \gamma_{21} \approx $ `[ ... ]`.
- **Tổng trách nhiệm:** $N_1 = $ `[ ... ]`, $N_2 = $ `[ ... ]`
- **Kỳ vọng mới $\mu_1^{new}$:** `[ ... ]`



In [ ]:
# D. Ô KIỂM TRA ĐÁP ÁN BÀI TẬP 4 (Chạy cell này để kiểm tra kết quả tính tay của bạn)
X_sv4 = np.array([1.0, 2.0, 5.0, 6.0])
pi_init = np.array([0.5, 0.5])
mu_init = np.array([1.5, 5.5])
var_init = np.array([1.0, 1.0])

# E-step
pdf1 = norm.pdf(X_sv4, loc=mu_init[0], scale=np.sqrt(var_init[0]))
pdf2 = norm.pdf(X_sv4, loc=mu_init[1], scale=np.sqrt(var_init[1]))
numer = np.column_stack([pi_init[0] * pdf1, pi_init[1] * pdf2])
gamma_mat = numer / np.sum(numer, axis=1, keepdims=True)

# M-step
N_res = np.sum(gamma_mat, axis=0)
pi_res = N_res / len(X_sv4)
mu_res = np.sum(gamma_mat * X_sv4[:, np.newaxis], axis=0) / N_res
var_res = np.sum(gamma_mat * (X_sv4[:, np.newaxis] - mu_res[np.newaxis, :])**2, axis=0) / N_res

print("====== ĐÁP ÁN CHUẨN ĐỐI CHIẾU BÀI 4 ======")
print("Ma trận trách nhiệm gamma (E-step):")
print(np.round(gamma_mat, 4))
print(f"\nTổng trách nhiệm hiệu dụng N_k: {N_res}")
print(f"Trọng số cụm mới pi: {pi_res}")
print(f"Kỳ vọng mới mu: {np.round(mu_res, 4)} (Kỳ vọng: [1.5038, 5.4962])")
print(f"Phương sai mới var: {np.round(var_res, 4)} (Kỳ vọng: [0.2653, 0.2653])")


## 3. Trực Quan Hóa Minh Họa Lý Thuyết Chuyên Sâu

Các khối code dưới đây minh họa trực quan các đặc trưng hình thái học của từng thuật toán. Sinh viên hãy quan sát kỹ các biểu đồ được tạo ra.



In [ ]:
# Trực quan hóa 1: Biên phân vùng Voronoi và quỹ đạo hội tụ K-Means
from scipy.spatial import Voronoi, voronoi_plot_2d

X_blobs, _ = make_blobs(n_samples=300, centers=[[-3, -2], [2, 4], [4, -3]], cluster_std=0.8, random_state=42)

# Tâm khởi tạo ngẫu nhiên (Epoch 0)
km_init = KMeans(n_clusters=3, init='random', n_init=1, max_iter=1, random_state=12).fit(X_blobs)
init_c = km_init.cluster_centers_

# Tâm hội tụ hoàn chỉnh
km_opt = KMeans(n_clusters=3, init='k-means++', n_init=10, random_state=42).fit(X_blobs)
opt_c = km_opt.cluster_centers_

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Đồ thị Voronoi
vor = Voronoi(opt_c)
voronoi_plot_2d(vor, ax=axes[0], show_vertices=False, line_colors='crimson', line_width=2, line_style='--')
axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], c=km_opt.labels_, cmap='viridis', alpha=0.5, s=25)
axes[0].scatter(opt_c[:, 0], opt_c[:, 1], c='red', marker='X', s=200, edgecolors='black', label='Tâm cụm hội tụ')
axes[0].set_title("Biên Phân vùng Voronoi (Voronoi Tessellation)", fontsize=12, fontweight='bold')
axes[0].legend()

# Đồ thị quỹ đạo dịch chuyển tâm
axes[1].scatter(X_blobs[:, 0], X_blobs[:, 1], c='gray', alpha=0.25, s=20)
axes[1].scatter(init_c[:, 0], init_c[:, 1], c='orange', marker='o', s=160, edgecolors='black', label='Tâm ban đầu (Epoch 0)')
axes[1].scatter(opt_c[:, 0], opt_c[:, 1], c='red', marker='X', s=200, edgecolors='black', label='Tâm cuối (Convergence)')
for c0, c1 in zip(init_c, opt_c):
    axes[1].annotate('', xy=c1, xytext=c0, arrowprops=dict(facecolor='black', edgecolor='black', width=1.5, headwidth=8))
axes[1].set_title("Quỹ đạo Dịch chuyển Trọng tâm qua các Epoch", fontsize=12, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Trực quan hóa 2: Cây Phân cấp Dendrogram & Đường Cắt Ngang Cụm
sample_idx = np.random.choice(len(X_blobs), size=25, replace=False)
X_sub = X_blobs[sample_idx]
Z_sub = linkage(X_sub, method='ward')

plt.figure(figsize=(11, 4.5))
dendrogram(Z_sub, leaf_rotation=90, leaf_font_size=10, color_threshold=7.0)
plt.axhline(y=7.0, color='crimson', linestyle='--', linewidth=2, label='Đường cắt ngang h = 7.0 (Phân tách thành 3 cụm)')
plt.title("Biểu đồ Cây Dendrogram (Ward Linkage)", fontsize=12, fontweight='bold')
plt.xlabel("Chỉ số Mẫu dữ liệu (Sample Index)", fontsize=10)
plt.ylabel("Khoảng cách Hợp nhất Ward", fontsize=10)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
# Trực quan hóa 3: Đồ thị K-Distance Elbow xác định Epsilon cho DBSCAN
X_moons, _ = make_moons(n_samples=400, noise=0.08, random_state=42)

k_kth = 4
nbrs = NearestNeighbors(n_neighbors=k_kth).fit(X_moons)
dists, _ = nbrs.kneighbors(X_moons)
sorted_dists = np.sort(dists[:, k_kth - 1])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Đồ thị K-Distance
axes[0].plot(sorted_dists, color='navy', linewidth=2.5)
axes[0].axhline(y=0.18, color='crimson', linestyle='--', linewidth=2, label='Điểm uốn khuỷu tay (Eps tối ưu = 0.18)')
axes[0].set_title("Đồ thị K-Distance Plot (k = 4)", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Các mẫu sắp xếp theo khoảng cách tăng dần", fontsize=10)
axes[0].set_ylabel("Khoảng cách tới láng giềng thứ 4", fontsize=10)
axes[0].legend(fontsize=10)

# Phân cụm DBSCAN
db_res = DBSCAN(eps=0.18, min_samples=5).fit(X_moons)
c_map = ['red' if l == -1 else plt.cm.Set1(l) for l in db_res.labels_]
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=c_map, s=25, alpha=0.8)
axes[1].set_title("Kết quả DBSCAN trên Dữ liệu Trăng khuyết Phi tuyến", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# Trực quan hóa 4: Elip Tin cậy của Mô hình GMM (1-sigma, 2-sigma, 3-sigma)
from matplotlib.patches import Ellipse

def plot_gmm_ellipse(pos, cov, ax=None, **kwargs):
    ax = ax or plt.gca()
    if cov.shape == (2, 2):
        U, s, _ = np.linalg.svd(cov)
        ang = np.degrees(np.arctan2(U[1, 0], U[0, 0]))
        w, h = 2 * np.sqrt(s)
    else:
        ang = 0
        w, h = 2 * np.sqrt(cov)
    for n_sig in [1, 2, 3]:
        ax.add_patch(Ellipse(pos, n_sig * w, n_sig * h, angle=ang, **kwargs))

np.random.seed(42)
X_el1 = np.random.randn(200, 2) @ [[1.5, 0.8], [0.2, 0.5]] + [2, 3]
X_el2 = np.random.randn(200, 2) @ [[0.5, -0.9], [1.2, 0.3]] + [-2, -1]
X_el = np.vstack([X_el1, X_el2])

gmm_viz = GaussianMixture(n_components=2, covariance_type='full', random_state=42).fit(X_el)

plt.figure(figsize=(8, 5.5))
plt.scatter(X_el[:, 0], X_el[:, 1], c=gmm_viz.predict(X_el), cmap='viridis', s=20, alpha=0.5)
plt.scatter(gmm_viz.means_[:, 0], gmm_viz.means_[:, 1], c='red', marker='X', s=180, label='Kỳ vọng tâm cụm')

for m, c in zip(gmm_viz.means_, gmm_viz.covariances_):
    plot_gmm_ellipse(m, c, alpha=0.2, facecolor='orange', edgecolor='red', linewidth=1.5)

plt.title("Elip Đồng Mức Tin cậy (1-sigma, 2-sigma, 3-sigma) của GMM", fontsize=12, fontweight='bold')
plt.xlabel("Trục X1")
plt.ylabel("Trục X2")
plt.legend()
plt.tight_layout()
plt.show()


## 4. Bài tập Lập trình Điền Khuyết (Fill-in-the-Blank Code Labs)

> **Hướng dẫn làm bài:**
> - Các vị trí cần điền code được ký hiệu bằng `...`.
> - Hãy đọc kỹ mô tả `TODO` và phần gợi ý để thay thế `...` bằng lệnh Python chuẩn.
> - Sau khi điền xong mỗi bài Lab, chạy ô test `assert` ngay bên dưới để chấm điểm tự động.

### Chuẩn bị Bộ dữ liệu Khách hàng Thực hành (Customer Segmentation Dataset)
Bộ dữ liệu gồm 250 khách hàng với 4 thuộc tính mua sắm:
- `Annual_Income_k$`: Thu nhập hàng năm (nghìn USD).
- `Spending_Score`: Điểm chi tiêu (thang 1 - 100).
- `Loyalty_Years`: Số năm gắn bó của khách hàng.
- `Purchase_Frequency`: Số lượt mua sắm mỗi tháng.



In [ ]:
# Tạo bộ dữ liệu khách hàng thực hành
np.random.seed(42)
c1 = np.random.randn(60, 4) * [8, 8, 1.0, 1.5] + [30, 20, 1.5, 3]    # Tiết kiệm thu nhập thấp
c2 = np.random.randn(70, 4) * [10, 9, 1.2, 2.0] + [35, 80, 2.0, 8]   # Chi tiêu phóng khoáng
c3 = np.random.randn(60, 4) * [9, 7, 1.5, 2.0] + [90, 25, 4.0, 4]    # Thu nhập cao, cẩn trọng
c4 = np.random.randn(60, 4) * [12, 10, 1.8, 2.5] + [95, 85, 5.5, 12] # Khách hàng VIP

df_customers = pd.DataFrame(
    np.vstack([c1, c2, c3, c4]),
    columns=['Annual_Income_k$', 'Spending_Score', 'Loyalty_Years', 'Purchase_Frequency']
)

print(f"Kích thước dataset khách hàng: {df_customers.shape}")
df_customers.head()


---

### Lab 1: K-Means Clustering & Phương pháp Khuỷu tay (Elbow)

**Yêu cầu:**
1. Chuẩn hóa `df_customers` bằng `StandardScaler().fit_transform()`.
2. Khảo sát $K \in [2, 8]$, lưu `.inertia_` và `silhouette_score`.
3. Huấn luyện mô hình K-Means tối ưu với $K = 4$.



In [ ]:
# BÀI TẬP LAB 1: SINH VIÊN THAY THẾ CÁC VỊ TRÍ '...' BẰNG CODE ĐÚNG

# Bước 1: Chuẩn hóa toàn bộ dữ liệu khách hàng
scaler = StandardScaler()
# TODO 1.1: Gọi fit_transform trên df_customers và gán vào X_scaled
X_scaled = ...

# Bước 2: Vòng lặp khảo sát K từ 2 đến 8
k_range = range(2, 9)
inertias = []
silhouette_scores = []

for k in k_range:
    # TODO 1.2: Khởi tạo KMeans với n_clusters=k, init='k-means++', n_init=10, random_state=42
    km = ...
    
    # TODO 1.3: Fit mô hình km vào dữ liệu X_scaled (chỉ điền phần gọi phương thức)
    ...
    
    # TODO 1.4: Lấy thuộc tính inertia_ của km và thêm vào danh sách inertias
    ...
    
    # TODO 1.5: Tính silhouette_score(X_scaled, km.labels_) và gán vào biến score
    score = ...
    silhouette_scores.append(score)

# Bước 3: Huấn luyện mô hình tối ưu với K = 4
# TODO 1.6: Khởi tạo KMeans(n_clusters=4, init='k-means++', n_init=10, random_state=42)
best_kmeans = ...
# Gọi fit_predict trên X_scaled để gán nhãn dự báo vào kmeans_labels
kmeans_labels = ...

print("Đã chạy xong cell Lab 1! Hãy chạy tiếp cell assert bên dưới để chấm điểm.")


In [ ]:
# KIỂM TRA ĐÁNH GIÁ TỰ ĐỘNG LAB 1
if X_scaled is ... or kmeans_labels is ...:
    print("⚠️ THÔNG BÁO: Bạn chưa điền đầy đủ code ở các vị trí '...' trong Lab 1.")
    print("👉 Hãy thay thế '...' bằng code đúng và chạy lại cell trên trước khi kiểm tra!")
else:
    assert X_scaled.shape == (250, 4), "Lỗi: X_scaled phải có kích thước (250, 4)!"
    assert np.allclose(X_scaled.mean(axis=0), 0, atol=1e-2), "Lỗi: Dữ liệu chưa được chuẩn hóa trung bình về 0!"
    assert len(inertias) == 7, "Lỗi: inertias phải có đúng 7 phần tử ứng với k từ 2 đến 8!"
    assert len(silhouette_scores) == 7, "Lỗi: silhouette_scores phải có đúng 7 phần tử!"
    assert inertias[0] > inertias[-1], "Lỗi: Inertia phải giảm khi tăng k!"
    assert len(set(kmeans_labels)) == 4, "Lỗi: kmeans_labels phải có đúng 4 nhãn cụm!"
    print("🎉 XUẤT SẮC! BẠN ĐÃ VƯỢT QUA TẤT CẢ TEST CASES CỦA LAB 1!")


---

### Lab 2: Phân cụm Phân cấp Thứ bậc (Hierarchical Clustering)

**Yêu cầu:**
1. Tính ma trận liên kết bằng hàm `linkage(..., method='ward')`.
2. Phân cụm 4 nhóm bằng `AgglomerativeClustering`.
3. Đánh giá chất lượng bằng chỉ số Davies-Bouldin Index (`davies_bouldin_score`).



In [ ]:
# BÀI TẬP LAB 2: SINH VIÊN THAY THẾ CÁC VỊ TRÍ '...' BẰNG CODE ĐÚNG

# TODO 2.1: Gọi hàm linkage() của SciPy trên X_scaled với method='ward'
Z_matrix = ...

# TODO 2.2: Khởi tạo mô hình AgglomerativeClustering với n_clusters=4, metric='euclidean', linkage='ward'
agg_cluster = ...

# TODO 2.3: Gọi fit_predict(X_scaled) trên agg_cluster và gán vào biến hierarchical_labels
hierarchical_labels = ...

# TODO 2.4: Gọi davies_bouldin_score(X_scaled, hierarchical_labels) và gán vào dbi_hierarchical
dbi_hierarchical = ...

print(f"Chỉ số Davies-Bouldin Index của Hierarchical: {dbi_hierarchical}")
print("Đã chạy xong cell Lab 2! Hãy chạy tiếp cell assert bên dưới để chấm điểm.")


In [ ]:
# KIỂM TRA ĐÁNH GIÁ TỰ ĐỘNG LAB 2
if Z_matrix is ... or hierarchical_labels is ...:
    print("⚠️ THÔNG BÁO: Bạn chưa điền đầy đủ code ở các vị trí '...' trong Lab 2.")
else:
    assert Z_matrix.shape == (249, 4), "Lỗi: Ma trận liên kết cho 250 mẫu phải có kích thước (249, 4)!"
    assert len(hierarchical_labels) == 250, "Lỗi: Số lượng nhãn phải bằng 250!"
    assert len(set(hierarchical_labels)) == 4, "Lỗi: Số lượng cụm phải bằng 4!"
    assert 0.0 < dbi_hierarchical < 1.5, "Lỗi: Giá trị Davies-Bouldin nằm ngoài dải kỳ vọng!"
    print("🎉 CHÚC MỪNG! BẠN ĐÃ VƯỢT QUA TẤT CẢ TEST CASES CỦA LAB 2!")


---

### Lab 3: DBSCAN & Tự động Nhận diện Điểm Nhiễu (Outliers)

**Yêu cầu:**
1. Dùng `NearestNeighbors` tìm khoảng cách đến láng giềng thứ 5.
2. Cấu hình `DBSCAN(eps=0.75, min_samples=5)` để phân cụm.
3. Đếm số cụm thực và số điểm nhiễu (mang nhãn `-1`).



In [ ]:
# BÀI TẬP LAB 3: SINH VIÊN THAY THẾ CÁC VỊ TRÍ '...' BẰNG CODE ĐÚNG

# TODO 3.1: Khởi tạo NearestNeighbors(n_neighbors=5) và fit vào X_scaled
nn_model = ...
distances, _ = nn_model.kneighbors(X_scaled) if nn_model is not ... else (None, None)

# TODO 3.2: Khởi tạo mô hình DBSCAN với eps=0.75 và min_samples=5
dbscan_model = ...

# TODO 3.3: Gọi fit_predict(X_scaled) trên dbscan_model và gán vào dbscan_labels
dbscan_labels = ...

# TODO 3.4: Đếm số lượng cụm phát hiện (loại trừ nhãn nhiễu -1)
# Gợi ý: len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_clusters_dbscan = ...

# TODO 3.5: Đếm tổng số điểm nhiễu (nhãn bằng -1)
# Gợi ý: np.sum(dbscan_labels == -1)
n_noise_points = ...

print(f"Số lượng cụm phát hiện: {n_clusters_dbscan}")
print(f"Số lượng điểm nhiễu: {n_noise_points}")
print("Đã chạy xong cell Lab 3! Hãy chạy tiếp cell assert bên dưới để chấm điểm.")


In [ ]:
# KIỂM TRA ĐÁNH GIÁ TỰ ĐỘNG LAB 3
if dbscan_labels is ... or n_clusters_dbscan is ...:
    print("⚠️ THÔNG BÁO: Bạn chưa điền đầy đủ code ở các vị trí '...' trong Lab 3.")
else:
    assert len(dbscan_labels) == 250, "Lỗi: Số lượng nhãn phải bằng 250!"
    assert n_clusters_dbscan >= 3, "Lỗi: DBSCAN phải tìm thấy ít nhất 3 cụm!"
    assert n_noise_points >= 0, "Lỗi: Số điểm nhiễu không hợp lệ!"
    assert -1 in dbscan_labels, "Lỗi: Tập dữ liệu phải có điểm ngoại lai nhận nhãn -1!"
    print("🎉 XUẤT SẮC! THUẬT TOÁN DBSCAN ĐÃ ĐƯỢC THỰC THI CHUẨN XÁC!")


---

### Lab 4: Gaussian Mixture Models (GMM) & Phân cụm Mềm

**Yêu cầu:**
1. Khảo sát $K \in [2, 8]$ và tính chỉ số $\text{BIC}(X)$ và $\text{AIC}(X)$.
2. Huấn luyện GMM tối ưu với $K = 4$ cụm.
3. Trích xuất ma trận xác suất thành viên mềm `soft_probs` (`predict_proba`) và nhãn cứng `gmm_labels` (`predict`).



In [ ]:
# BÀI TẬP LAB 4: SINH VIÊN THAY THẾ CÁC VỊ TRÍ '...' BẰNG CODE ĐÚNG

bic_list = []
aic_list = []
k_test_range = range(2, 9)

for k in k_test_range:
    # TODO 4.1: Khởi tạo GaussianMixture với n_components=k, covariance_type='full', random_state=42
    gmm_k = ...
    
    # TODO 4.2: Fit mô hình gmm_k vào X_scaled
    ...
    
    # TODO 4.3: Gọi gmm_k.bic(X_scaled) và gmm_k.aic(X_scaled) rồi append vào 2 danh sách
    ...
    ...

# Bước 2: Huấn luyện GMM tối ưu với K = 4
# TODO 4.4: Khởi tạo GaussianMixture(n_components=4, covariance_type='full', random_state=42)
best_gmm = ...
# Fit mô hình vào X_scaled
...

# TODO 4.5: Trích xuất ma trận xác suất mềm bằng best_gmm.predict_proba(X_scaled)
soft_probs = ...

# TODO 4.6: Trích xuất nhãn phân cụm cứng bằng best_gmm.predict(X_scaled)
gmm_labels = ...

print("Đã chạy xong cell Lab 4! Hãy chạy tiếp cell assert bên dưới để chấm điểm.")


In [ ]:
# KIỂM TRA ĐÁNH GIÁ TỰ ĐỘNG LAB 4
if soft_probs is ... or gmm_labels is ...:
    print("⚠️ THÔNG BÁO: Bạn chưa điền đầy đủ code ở các vị trí '...' trong Lab 4.")
else:
    assert len(bic_list) == 7 and len(aic_list) == 7, "Lỗi: Danh sách BIC/AIC phải có đúng 7 giá trị!"
    assert soft_probs.shape == (250, 4), "Lỗi: Ma trận xác suất mềm phải có kích thước (250, 4)!"
    assert np.allclose(np.sum(soft_probs, axis=1), 1.0), "Lỗi: Tổng xác suất các cụm của mỗi điểm phải bằng 1.0!"
    assert len(set(gmm_labels)) == 4, "Lỗi: GMM phải dự báo 4 cụm!"
    print("🎉 HOÀN HẢO! BẠN ĐÃ LÀM CHỦ PHÂN CỤM XÁC SUẤT GMM VÀ THUẬT TOÁN EM!")


---

### Lab 5: Bảng Xếp hạng Benchmark So sánh các Thuật toán

Chạy ô lệnh dưới đây sau khi đã hoàn thành các Lab 1, 2, 3, 4 để so sánh trực diện chất lượng phân cụm:



In [ ]:
# So sánh tổng hợp 4 thuật toán trên dữ liệu khách hàng
try:
    valid_mask = dbscan_labels != -1

    models_comparison = {
        "Thuật toán": ["K-Means (Centroid)", "Hierarchical (Ward)", "DBSCAN (Density)", "GMM (Distribution)"],
        "Silhouette Score (↑)": [
            silhouette_score(X_scaled, kmeans_labels),
            silhouette_score(X_scaled, hierarchical_labels),
            silhouette_score(X_scaled[valid_mask], dbscan_labels[valid_mask]),
            silhouette_score(X_scaled, gmm_labels)
        ],
        "Davies-Bouldin Index (↓)": [
            davies_bouldin_score(X_scaled, kmeans_labels),
            davies_bouldin_score(X_scaled, hierarchical_labels),
            davies_bouldin_score(X_scaled[valid_mask], dbscan_labels[valid_mask]),
            davies_bouldin_score(X_scaled, gmm_labels)
        ],
        "Calinski-Harabasz Index (↑)": [
            calinski_harabasz_score(X_scaled, kmeans_labels),
            calinski_harabasz_score(X_scaled, hierarchical_labels),
            calinski_harabasz_score(X_scaled[valid_mask], dbscan_labels[valid_mask]),
            calinski_harabasz_score(X_scaled, gmm_labels)
        ]
    }

    df_benchmark = pd.DataFrame(models_comparison).set_index("Thuật toán")
    print("====== BẢNG TỔNG HỢP SO SÁNH CHẤT LƯỢNG PHÂN CỤM ======")
    display(df_benchmark.round(4))
except Exception as e:
    print("⚠️ Hãy hoàn thành đầy đủ cả 4 bài Lab 1, 2, 3, 4 trước khi chạy bảng so sánh!")


## 5. Đáp án Tham khảo Code Điền khuyết (Reference Solutions)

Dưới đây là đáp án hoàn chỉnh cho toàn bộ các bài tập Lab để Sinh viên đối chiếu khi gặp khó khăn:

```python
# =============================================================================
# ĐÁP ÁN LAB 1: K-MEANS
# =============================================================================
X_scaled = scaler.fit_transform(df_customers)

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    score = silhouette_score(X_scaled, km.labels_)
    silhouette_scores.append(score)

best_kmeans = KMeans(n_clusters=4, init='k-means++', n_init=10, random_state=42)
kmeans_labels = best_kmeans.fit_predict(X_scaled)

# =============================================================================
# ĐÁP ÁN LAB 2: HIERARCHICAL CLUSTERING
# =============================================================================
Z_matrix = linkage(X_scaled, method='ward')
agg_cluster = AgglomerativeClustering(n_clusters=4, metric='euclidean', linkage='ward')
hierarchical_labels = agg_cluster.fit_predict(X_scaled)
dbi_hierarchical = davies_bouldin_score(X_scaled, hierarchical_labels)

# =============================================================================
# ĐÁP ÁN LAB 3: DBSCAN
# =============================================================================
nn_model = NearestNeighbors(n_neighbors=5).fit(X_scaled)
distances, _ = nn_model.kneighbors(X_scaled)
dbscan_model = DBSCAN(eps=0.75, min_samples=5)
dbscan_labels = dbscan_model.fit_predict(X_scaled)
n_clusters_dbscan = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise_points = np.sum(dbscan_labels == -1)

# =============================================================================
# ĐÁP ÁN LAB 4: GMM & EM
# =============================================================================
for k in k_test_range:
    gmm_k = GaussianMixture(n_components=k, covariance_type='full', random_state=42)
    gmm_k.fit(X_scaled)
    bic_list.append(gmm_k.bic(X_scaled))
    aic_list.append(gmm_k.aic(X_scaled))

best_gmm = GaussianMixture(n_components=4, covariance_type='full', random_state=42)
best_gmm.fit(X_scaled)
soft_probs = best_gmm.predict_proba(X_scaled)
gmm_labels = best_gmm.predict(X_scaled)
```

---
**Chúc mừng bạn đã hoàn thành trọn vẹn bài thực hành Phân cụm Dữ liệu!**

